In [ ]:
from pyspark.sql import functions as F
from infra.spark_utils import build_spark, normalize_ptbr_number
from infra.tickers_cache import get_tickers_price, handler_tickers_cache, handler_partitions

In [2]:
import sys
print(sys.executable)

c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe


In [3]:
spark = build_spark()

In [4]:
import sys
import os

print("Notebook Python:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))

Notebook Python: c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe
PYSPARK_PYTHON: c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON: c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe


In [5]:
spark = build_spark()
file_path = "../data/bronze/forms/economias.csv"
df = spark.read.csv(
    path = file_path,
    sep = ",", 
    header=True,
    multiLine=True # se n tiver quebra a logica 
)
df

DataFrame[timestamp: string, data_apuracao: string, instituicao_fin: string, resumo: string, aporte: string]

In [6]:
df = (df.withColumn(
    'resumo',
    F.regexp_replace(F.col('resumo'), r"\r\n|\r", "\n")
    )
    .withColumn(
    'resumo',
    F.split(F.col('resumo'), '\n')
    )
    .withColumn(
            'resumo',
            F.explode(F.col('resumo'))
        )
)
df.show()

+-------------------+-------------+---------------+--------------------+------+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|
+-------------------+-------------+---------------+--------------------+------+
|24/02/2026 10:21:10|   14/03/2026|             XP|Stock | BERK34 | ...|     0|
|24/02/2026 10:21:10|   14/03/2026|             XP|Renda Fixa | Teso...|     0|
|24/02/2026 10:22:11|   14/03/2026|          Clear|Stock | IVVB11 | ...|     0|
|24/02/2026 10:22:11|   14/03/2026|          Clear|Stock  | BOVA11 |...|     0|
|24/02/2026 10:22:11|   14/03/2026|          Clear|Stock | BERK34 | ...|     0|
|24/02/2026 10:23:26|   14/03/2026|         Nubank|Renda Fixa | Rese...|     0|
+-------------------+-------------+---------------+--------------------+------+



In [7]:
parts = F.split(F.col('resumo'), r"\|")
df = (df
      .withColumn("tipo", F.trim(parts.getItem(0)))
      .withColumn("nome", F.trim(parts.getItem(1)))
      .withColumn("qtd", F.trim(parts.getItem(2)))
      .withColumn("preco_medio", F.trim(parts.getItem(3)))
      .withColumn("preco_atual", F.trim(parts.getItem(4)))
      .withColumn("tipo", normalize_ptbr_number(F.col("tipo")))
      .withColumn("nome", normalize_ptbr_number(F.col("nome")))
      .withColumn("qtd", normalize_ptbr_number(F.col("qtd")))
      .withColumn("preco_medio", normalize_ptbr_number(F.col("preco_medio")))
      .withColumn("preco_atual", normalize_ptbr_number(F.col("preco_atual")))
    )

df.show()

+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|      tipo|                nome|qtd|preco_medio|preco_atual|
+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+
|24/02/2026 10:21:10|   14/03/2026|             XP|Stock | BERK34 | ...|     0|     Stock|              BERK34|  5|     120.02|       NULL|
|24/02/2026 10:21:10|   14/03/2026|             XP|Renda Fixa | Teso...|     0|Renda Fixa|             Tesouro|  1|   24000.00|   30000.00|
|24/02/2026 10:22:11|   14/03/2026|          Clear|Stock | IVVB11 | ...|     0|     Stock|              IVVB11| 15|     300.05|       NULL|
|24/02/2026 10:22:11|   14/03/2026|          Clear|Stock  | BOVA11 |...|     0|     Stock|              BOVA11| 10|      125.2|       NULL|
|24/02/2026 10:22:11

In [8]:
mapping_month = F.create_map(
    F.lit(1), F.lit("Jan"),  
    F.lit(2), F.lit("Fev"),  
    F.lit(3), F.lit("Mar"),  
    F.lit(4), F.lit("Abr"),  
    F.lit(5), F.lit("Mai"),  
    F.lit(6), F.lit("Jun"),  
    F.lit(7), F.lit("Jul"),  
    F.lit(8), F.lit("Ago"),  
    F.lit(9), F.lit("Set"),  
    F.lit(10), F.lit("Out"), 
    F.lit(11), F.lit("Nov"),
    F.lit(12), F.lit("Dec")
)

In [9]:
df = (df
      .withColumn(
          'data_apuracao',
          F.to_date(F.col('data_apuracao'), format='dd/MM/yyyy')
      )
      .withColumn(
          'ano',
          F.year(F.col('data_apuracao'))
      )
      .withColumn(
          'ano',
          F.year(F.col('data_apuracao'))
      )
      .withColumn(
          'mes_num',
          F.month(F.col('data_apuracao'))
      )
      .withColumn(
          'mes',
          mapping_month[F.col('mes_num')]
      )
)

df.show()


+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|      tipo|                nome|qtd|preco_medio|preco_atual| ano|mes_num|mes|
+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+
|24/02/2026 10:21:10|   2026-03-14|             XP|Stock | BERK34 | ...|     0|     Stock|              BERK34|  5|     120.02|       NULL|2026|      3|Mar|
|24/02/2026 10:21:10|   2026-03-14|             XP|Renda Fixa | Teso...|     0|Renda Fixa|             Tesouro|  1|   24000.00|   30000.00|2026|      3|Mar|
|24/02/2026 10:22:11|   2026-03-14|          Clear|Stock | IVVB11 | ...|     0|     Stock|              IVVB11| 15|     300.05|       NULL|2026|      3|Mar|
|24/02/2026 10:22:11|   2026-03-14|          Clear|Stock  

In [10]:
tickers_internacionais = ['BERK34', 'IVVB11', 'AAPL34', 'MSFT34', 'NVDC34', 'GOGL34', 'AMZO34', 'TSLA34', 'META34', 'MELI34']

df = (df
      .withColumn(
          'exposicao', 
          F.when(F.col('nome').isin(tickers_internacionais), "internacional")
          .otherwise("nacional")
      )
      .withColumn(
          'tipo',
          F.lower(F.col('tipo'))
      )
      .withColumn(
          'instituicao_fin',
          F.upper(F.col('instituicao_fin'))
      )
      .withColumn(
          'tipo',
          F.lower(F.col('tipo'))
      )
      .withColumn(
          'nome',
          F.when(F.col('tipo') == 'stock', F.upper(F.col('nome')))
          .otherwise(F.lower(F.col('nome')))
      )
)

df.show()

+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+-------------+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|      tipo|                nome|qtd|preco_medio|preco_atual| ano|mes_num|mes|    exposicao|
+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+-------------+
|24/02/2026 10:21:10|   2026-03-14|             XP|Stock | BERK34 | ...|     0|     stock|              BERK34|  5|     120.02|       NULL|2026|      3|Mar|internacional|
|24/02/2026 10:21:10|   2026-03-14|             XP|Renda Fixa | Teso...|     0|renda fixa|             tesouro|  1|   24000.00|   30000.00|2026|      3|Mar|     nacional|
|24/02/2026 10:22:11|   2026-03-14|          CLEAR|Stock | IVVB11 | ...|     0|     stock|              IVVB11| 15|     300.05|       NULL|2026| 

In [11]:
df = (df.select(
    F.to_timestamp(F.col('timestamp'), "dd/MM/yyyy HH:mm:ss").alias('timestamp'),
    F.to_date(F.col('data_apuracao'), "dd/MM/yyyy").alias('data_apuracao'),
    F.col('ano').cast('int').alias('ano'),
    F.col('mes_num').cast('int').alias('mes_num'),
    F.col('mes').cast('string').alias('mes'),
    F.col('instituicao_fin').cast('string').alias('instituicao_fin'),
    F.col('resumo').cast('string').alias('resumo'),
    F.col('tipo').cast('string').alias('tipo'),
    F.col('nome').cast('string').alias('nome'),
    F.col('qtd').cast('double').alias('qtd'),
    F.col('preco_medio').cast('double').alias('preco_medio'),
    F.col('preco_atual').cast('double').alias('preco_atual'),
    F.col('aporte').cast('double').alias('aporte'),
    F.col('exposicao').cast('string').alias('exposicao')
    )
)
df.show()

+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|          timestamp|data_apuracao| ano|mes_num|mes|instituicao_fin|              resumo|      tipo|                nome| qtd|preco_medio|preco_atual|aporte|    exposicao|
+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Stock | BERK34 | ...|     stock|              BERK34| 5.0|     120.02|       NULL|   0.0|internacional|
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Renda Fixa | Teso...|renda fixa|             tesouro| 1.0|    24000.0|    30000.0|   0.0|     nacional|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | IVVB11 | ...|     stock|              IVVB11|15.0|     300.05|  

In [12]:
df_price = get_tickers_price(df)
df_price.show()

[*********************100%***********************]  3 of 3 completed


+----------+------+------+--------------------+-------------+
|data_preco|ticker| close|        extracted_at|data_apuracao|
+----------+------+------+--------------------+-------------+
|2026-03-13|BERK34|130.33|2026-03-14 09:57:...|   2026-03-14|
|2026-03-13|BOVA11|174.55|2026-03-14 09:57:...|   2026-03-14|
|2026-03-13|IVVB11| 397.3|2026-03-14 09:57:...|   2026-03-14|
+----------+------+------+--------------------+-------------+



In [13]:
df_cache = handler_tickers_cache(df_price)
df_cache.show()

+----------+------+------+--------------------+-------------+
|data_preco|ticker| close|        extracted_at|data_apuracao|
+----------+------+------+--------------------+-------------+
|2026-03-06|BERK34|130.05|2026-03-08 13:03:...|   2026-03-08|
|2026-03-07|BERK34|132.45| 2026-03-08 12:31:00|   2026-03-08|
|2026-03-13|BERK34|130.33|2026-03-14 09:57:...|   2026-03-14|
|2026-03-06|BOVA11|175.89|2026-03-08 13:03:...|   2026-03-08|
|2026-03-07|BOVA11|177.35| 2026-03-08 12:32:00|   2026-03-08|
|2026-03-13|BOVA11|174.55|2026-03-14 09:57:...|   2026-03-14|
|2026-03-06|IVVB11|397.87|2026-03-08 13:03:...|   2026-03-08|
|2026-03-07|IVVB11|399.12| 2026-03-08 12:33:00|   2026-03-08|
|2026-03-13|IVVB11| 397.3|2026-03-14 09:57:...|   2026-03-14|
|2026-03-07| PETR4| 38.72| 2026-03-08 12:34:00|   2026-03-08|
+----------+------+------+--------------------+-------------+



In [14]:
df = (df.alias('a')
      .join(
          df_cache.alias('b'), 
          how='left', 
          on=(
              (F.col('a.data_apuracao') == F.col('b.data_apuracao')) & # usando apuracao no join, tem q rodar no mesmo dia q foi apurado.
              (F.col('a.nome') == F.col('b.ticker'))
              )
        )
        .withColumn(
            'preco_atual',
            F.when(F.col('a.preco_atual').isNull(), F.col('b.close'))
            .otherwise(F.col('a.preco_atual'))
        )
        .select(
            F.to_timestamp(F.col('a.timestamp'), "dd/MM/yyyy HH:mm:ss").alias('timestamp'),
            F.to_date(F.col('a.data_apuracao'), "dd/MM/yyyy").alias('data_apuracao'),
            F.col('a.ano').cast('int').alias('ano'),
            F.col('a.mes_num').cast('int').alias('mes_num'),
            F.col('a.mes').cast('string').alias('mes'),
            F.col('a.instituicao_fin').cast('string').alias('instituicao_fin'),
            F.col('a.resumo').cast('string').alias('resumo'),
            F.col('a.tipo').cast('string').alias('tipo'),
            F.col('a.nome').cast('string').alias('nome'),
            F.col('a.qtd').cast('double').alias('qtd'),
            F.col('a.preco_medio').cast('double').alias('preco_medio'),
            F.col('preco_atual').cast('double').alias('preco_atual'),
            F.col('a.aporte').cast('double').alias('aporte'),
            F.col('a.exposicao').cast('string').alias('exposicao'),
        )
)

df.show()

+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|          timestamp|data_apuracao| ano|mes_num|mes|instituicao_fin|              resumo|      tipo|                nome| qtd|preco_medio|preco_atual|aporte|    exposicao|
+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Stock | BERK34 | ...|     stock|              BERK34| 5.0|     120.02|     130.33|   0.0|internacional|
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Renda Fixa | Teso...|renda fixa|             tesouro| 1.0|    24000.0|    30000.0|   0.0|     nacional|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | IVVB11 | ...|     stock|              IVVB11|15.0|     300.05|  

In [15]:
# n vou usar funcao para salvar particao agr
# df_snapshot = handler_partitions(df, 'silver')

# print(df_snapshot)

In [ ]:
df.createOrReplaceTempView("df_silver")

# somente os ultimos valores de cada investimento para pegar os valores atuais
query_valores_atuais = """
    SELECT 
        *
    FROM (
        SELECT 
            *
            , ROW_NUMBER() OVER(PARTITION BY instituicao_fin, tipo, nome ORDER BY data_apuracao DESC) AS rn
        FROM df_silver
    )
    WHERE rn = 1
"""

df_gold_val_atual = spark.sql(query_valores_atuais)

df_gold_val_atual.show()

+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+---+
|          timestamp|data_apuracao| ano|mes_num|mes|instituicao_fin|              resumo|      tipo|                nome| qtd|preco_medio|preco_atual|aporte|    exposicao| rn|
+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+---+
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | BERK34 | ...|     stock|              BERK34|10.0|     132.29|     130.33|   0.0|internacional|  1|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock  | BOVA11 |...|     stock|              BOVA11|10.0|      125.2|     174.55|   0.0|     nacional|  1|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | IVVB11 | ...|     stock|              IVVB11

In [17]:
spark.stop()